# Post-processing

1) [Absurd values removal](#absurd-values-removal)
2) [Save results to gpkg](#save-results-to-gpkg)
3) [Count number of absurds per level](#count-number-of-absurds-per-level)

In [33]:
# dependencies
import os
import sys
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import pickle
from tqdm import tqdm
import open3d as o3d
import laspy
from plyfile import PlyData, PlyElement
import rasterio
from rasterio.transform import from_origin
from copy import deepcopy

import geopandas as gpd
from shapely.geometry import Point, Polygon
import pandas as pd
from time import time

sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
# print(sys.path)
from src.quadnode import QuadNode

### Absurd values removal

Load the geopackage output and find absurde values. Create then a status code as followed:
- 0: correct value
- 1: absurd value
- 2: correct value but sibling to an absurd value

In [29]:
def get_translate_from_element(bbox_dict, transform):
    xmin, ymin, zmin = bbox_dict['min_bound']
    xmax, ymax, zmax = bbox_dict['max_bound']
    center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
    translated = np.linalg.matmul(transform, center)
    norm = float(np.linalg.norm(translated - center))

    return norm


def compute_translation(bbox_dict, transform):
    xmin, ymin, zmin = bbox_dict['min_bound']
    xmax, ymax, zmax = bbox_dict['max_bound']
    center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
    translated = np.linalg.matmul(transform, center)
    norm = float(np.linalg.norm(translated - center))
    norm2d = float(np.linalg.norm(translated[0:2] - center[0:2]))
    # list_texts.append((cx, cy, norm, el_lvl))
    direction = (translated[0:2] - center[0:2]) / norm2d 

    return norm, direction


def trim_branch(node):
    if node.children != []:
        for child in node.children:
            trim_branch(child)
            
    # Detach from parent
    if node.parent is not None and node in node.parent.children:
        node.parent.children.remove(node)
        if node.parent.children == []:
            node.parent.is_leaf = True

    # Break all references so pickle can't follow them
    node.parent = None
    node.children = []
    

def detect_absurds(node):
    counter = 0
    norm, direction = compute_translation(node.bbox, node.transform)
    node.metrics['translation_direction'] = direction
    node.metrics['translation_norm'] = norm
    diff = 0

    if node.level > 0:
        parent = node.parent
        parent_norm, _ = compute_translation(parent.bbox, parent.transform)
        diff = abs(parent_norm - norm)

    if diff > 10:
        counter += 1
        for child in node.children:
            trim_branch(child)
        node.metrics['translation_direction'] = parent.metrics['translation_direction']
        node.metrics['translation_norm'] = parent.metrics['translation_norm']
        node.is_absurd = True
        node.is_leaf = True
        for child in node.children:
            child.parent = None
        node.children = []

    for child in node.children:
        counter += detect_absurds(child)
    return counter

In [57]:
# laoding of gpkg file
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_11_min_buffer_area\no_min\3000_max_lvl_7\results\pyramid_transforms_test.pickle"
src_results = src_transforms.split('.pickle')[0] + "_with_absurd_vals.pickle"
with open(src_transforms, 'rb') as f:
    root = pickle.load(f)

counter = detect_absurds(root)

print("Number of absurd values: ", counter)

with open(src_results, 'wb') as f:
    pickle.dump(root, f)

AttributeError: 'list' object has no attribute 'bbox'

### Save results to GPKG

In [55]:
def export_bboxes(data, columns, bbox_data, output_path, offset, layer_name, crs="EPSG:2056"):
    """
    Export points and bbox polygons as two layers in a single GPKG.

    Parameters:
        data: list of lists [x, y, lvl, val1, ..., valn]
        columns: column names for data
        bbox_data: list of dicts with keys 'min_bound', 'max_bound' and any attributes
        output_path: path with .gpkg extension
        crs: coordinate reference system (default: Swiss LV95)
    """

    df_points = pd.DataFrame(data, columns=columns)
    rows = []
    for bbox in bbox_data:
        minx, miny = bbox["min_bound"][0] + offset[0], bbox["min_bound"][1] + offset[1]
        maxx, maxy = bbox["max_bound"][0] + offset[0], bbox["max_bound"][1] + offset[1]
        poly = Polygon([
            (minx, miny),
            (maxx, miny),
            (maxx, maxy),
            (minx, maxy),
            (minx, miny)
        ])
        row = {k: v for k, v in bbox.items() if k not in ("min_bound", "max_bound")}
        row["geometry"] = poly
        rows.append(poly)

    gdf_bboxes = gpd.GeoDataFrame(df_points, geometry=rows, crs=crs)
    gdf_bboxes.to_file(output_path, layer=layer_name, driver="GPKG")  # append to same file


def export_points_and_bboxes(data, columns, bbox_data, output_path, offset, crs="EPSG:2056"):
    """
    Export points and bbox polygons as two layers in a single GPKG.

    Parameters:
        data: list of lists [x, y, lvl, val1, ..., valn]
        columns: column names for data
        bbox_data: list of dicts with keys 'min_bound', 'max_bound' and any attributes
        output_path: path with .gpkg extension
        crs: coordinate reference system (default: Swiss LV95)
    """

    # --- Layer 1: Points ---
    # apply offset
    df_points = pd.DataFrame(data, columns=columns)
    geometry_points = [Point(xy) for xy in zip(df_points["x"], df_points["y"])]
    gdf_points = gpd.GeoDataFrame(df_points, geometry=geometry_points, crs=crs)
    gdf_points.to_file(output_path, layer="displacements", driver="GPKG")

    # --- Layer 2: BBoxes ---
    rows = []
    for bbox in bbox_data:
        minx, miny = bbox["min_bound"][0] + offset[0], bbox["min_bound"][1] + offset[1]
        maxx, maxy = bbox["max_bound"][0] + offset[0], bbox["max_bound"][1] + offset[1]
        poly = Polygon([
            (minx, miny),
            (maxx, miny),
            (maxx, maxy),
            (minx, maxy),
            (minx, miny)
        ])
        row = {k: v for k, v in bbox.items() if k not in ("min_bound", "max_bound")}
        row["geometry"] = poly
        rows.append(poly)

    gdf_bboxes = gpd.GeoDataFrame(df_points, geometry=rows, crs=crs)
    gdf_bboxes.to_file(output_path, layer="tiles", driver="GPKG")  # append to same file

    print(f"Saved: {output_path}")
    print(f"  Points : {len(gdf_points)}")
    print(f"  BBoxes : {len(gdf_bboxes)}")


# def compute_translation(bbox_dict, transform):
#     xmin, ymin, zmin = bbox_dict['min_bound']
#     xmax, ymax, zmax = bbox_dict['max_bound']
#     center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
#     translated = np.linalg.matmul(transform, center)
#     norm = float(np.linalg.norm(translated - center))
#     norm2d = float(np.linalg.norm(translated[0:2] - center[0:2]))
#     direction = (translated[0:2] - center[0:2]) / norm2d 

#     return norm, direction


def compute_data_for_gpkg(node, data, bbox_data, offset):
    if isinstance(node, QuadNode):
        # compute translation
        bbox_dict = node.bbox
        bbox_data.append(bbox_dict)

        # position:
        xmin, ymin, zmin = bbox_dict['min_bound']
        xmax, ymax, zmax = bbox_dict['max_bound']
        center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])

        data.append(
            (center[0][0] + offset[0], center[1][0] + offset[1], center[2][0] + offset[2], 
            node.metrics['translation_norm'], 
            node.metrics['translation_direction'][0][0], 
            node.metrics['translation_direction'][1][0], 
            node.level, node.is_leaf, node.is_absurd))

        for child in node.children:
            compute_data_for_gpkg(child, data, bbox_data, offset)
    elif isinstance(node, list):
        for el in node:
            # compute translation
            bbox_dict = el.bbox
            bbox_data.append(bbox_dict)

            # position:
            xmin, ymin, zmin = bbox_dict['min_bound']
            xmax, ymax, zmax = bbox_dict['max_bound']
            center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])

            data.append(
                (center[0][0] + offset[0], center[1][0] + offset[1], center[2][0] + offset[2], 
                el.metrics['translation_norm'], 
                el.metrics['translation_direction'][0][0], 
                el.metrics['translation_direction'][1][0], 
                el.level, el.is_leaf, el.is_absurd))


def tree_in_list(node, list_nodes):
    if node.level > len(list_nodes) - 1:
        list_nodes.append([])
    list_nodes[node.level].append(node)
    for child in node.children:
        tree_in_list(child, list_nodes)

In [56]:
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swamp_and_lake\2588_1168_lake\results\pyramid_transforms_test_with_absurd_vals.pickle"
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_11_min_buffer_area\min_16m\3000_max_lvl_7\results\pyramid_transforms_test_with_absurd_vals.pickle"
src_tile_original = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_7_different_max_correspondances\down_to_0.2\swisssurface3d_2018_2589-1169_2056_5728.ply"
src_out_gpkg = os.path.join(os.path.dirname(src_transforms), 'points_translate.gpkg')
src_out_gpkg_leaves = src_out_gpkg.split('.gpkg')[0] + "_leaves.gpkg"
src_out_gpkg_layers = src_out_gpkg.split('.gpkg')[0] + "_layers.gpkg"

tile_original = o3d.io.read_point_cloud(src_tile_original)
z_mean = tile_original.compute_mean_and_covariance()[0][2]
offset = [2589500, 1169500, z_mean]

with open(src_transforms, 'rb') as f:
    root = pickle.load(f)

data = []
bbox_data = []

compute_data_for_gpkg(root, data, bbox_data, offset)

columns = ['x', 'y', 'z', 'translation', 'translation_x', 'translation_y', 'lvl', 'is_leaf', 'absurd_status']

# Export all tiles
export_points_and_bboxes(
    data=data,
    bbox_data=bbox_data,
    columns=columns,
    output_path=src_out_gpkg,
    offset=offset,
)

# Export only leaves
mask_leaves = np.array([x[-2] for x in data], dtype=np.bool)
data_leaves = list(np.array(data)[mask_leaves])
bbox_data_leaves = list(np.array(bbox_data)[mask_leaves])

export_points_and_bboxes(
    data=data_leaves,
    bbox_data=bbox_data_leaves,
    columns=columns,
    output_path=src_out_gpkg_leaves,
    offset=offset,
)

# Layer by layer
list_lvls = []
tree_in_list(root, list_lvls)
for lvl in range(len(list_lvls)):
    print("level: ", lvl, ' - num subtiles: ', len(list_lvls[lvl]))

    data = []
    bbox_data = [] 
    compute_data_for_gpkg(list_lvls[lvl], data, bbox_data, offset)

    columns = ['x', 'y', 'z', 'translation', 'translation_x', 'translation_y', 'lvl', 'is_leaf', 'absurd_status']

    # Export all tiles
    export_bboxes(
        data=data,
        bbox_data=bbox_data,
        columns=columns,
        output_path=src_out_gpkg_layers,
        offset=offset,
        layer_name=f"Level {lvl}"
    )
    

Saved: D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_11_min_buffer_area\min_16m\3000_max_lvl_7\results\points_translate.gpkg
  Points : 9585
  BBoxes : 9585
Saved: D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_11_min_buffer_area\min_16m\3000_max_lvl_7\results\points_translate_leaves.gpkg
  Points : 7189
  BBoxes : 7189
level:  0  - num subtiles:  1
level:  1  - num subtiles:  4
level:  2  - num subtiles:  16
level:  3  - num subtiles:  64
level:  4  - num subtiles:  256
level:  5  - num subtiles:  1024
level:  6  - num subtiles:  4068
level:  7  - num subtiles:  4152


### Count number of absurds per level

In [73]:
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_11_min_buffer_area\min_16m\3000_max_lvl_7\results\pyramid_transforms_test_with_absurd_vals.pickle"

with open(src_transforms, 'rb') as f:
    root = pickle.load(f)
list_lvls = []
tree_in_list(root, list_lvls)
for lvl, data in enumerate(list_lvls):
    num_absurds = len([x for x in data if x.is_absurd == True])
    print("lvl ", lvl, ' - num absurds: ', num_absurds)

lvl  0  - num absurds:  0
lvl  1  - num absurds:  0
lvl  2  - num absurds:  0
lvl  3  - num absurds:  0
lvl  4  - num absurds:  0
lvl  5  - num absurds:  0
lvl  6  - num absurds:  9
lvl  7  - num absurds:  64


# TEMP

In [27]:
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swamp_and_lake\2588_1168_lake\results\pyramid_transforms_test_with_absurd_vals.pickle"
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swamp_and_lake\2588_1168_lake\results\pyramid_transforms_test.pickle"
src_tile_original = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_7_different_max_correspondances\down_to_0.2\swisssurface3d_2018_2589-1169_2056_5728.ply"
src_out_gpkg = os.path.join(os.path.dirname(src_transforms), 'points_translate.gpkg')

tile_original = o3d.io.read_point_cloud(src_tile_original)
z_mean = tile_original.compute_mean_and_covariance()[0][2]
offset = [2589500, 1169500, z_mean]

with open(src_transforms, 'rb') as f:
    root = pickle.load(f)

print(root.bbox)

{'min_bound': [-1500.0, -1500.0, -66.5149131177318], 'max_bound': [-500.0099999997765, -500.0100000000093, 207.30508688226814]}


# Old stuff

In [ ]:
# # laoding of gpkg file
# # src_gpkg = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_5_min_points\3000_max_lvl_7\results\points_translate.gpkg"
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\pyramid_transforms_test.pickle"
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swamp_and_lake\2588_1168_lake\results\pyramid_transforms_test.pickle"
# src_results = src_transforms.split('.pickle')[0] + "_with_absurd_vals.pickle"
# with open(src_transforms, 'rb') as f:
#     root = pickle.load(f)
# data = gpd.read_file(src_gpkg)
# data['is_absurd'] = np.zeros(len(data), dtype=np.bool)
# lst_absurds = np.zeros(len(lst_transforms), dtype=np.bool)
# lst_absurds = [False] * len(lst_transforms)
# print(len(lst_absurds))

# # loop over nodes:
# for id_el, node in tqdm(enumerate(lst_transforms), total=len(lst_transforms)):
#     # print(id_el)
#     # node = el[0]
#     bbox = node.bbox
#     transform = node.transform
#     parent = node.parent
#     # lvl_el = node.level

#     if node.level == 0:
#         continue
    
#     # compare to parent value
#     val = get_translate_from_element(bbox, transform)
#     # id_parent = id_el-1
    
#     # while(lst_transforms[id_parent][0] != node.level-1):
#     #     id_parent -= 1

#     parent_val = get_translate_from_element(parent.bbox, parent.transform)
#     diff = abs(parent_val - val)
#     if diff > 10 or lst_absurds[id_parent] == True:
#         lst_absurds[id_el] = True

# print("Number of absurd values: ", np.sum(lst_absurds))
# lst_transforms = [(*x,y) for x,y in zip(lst_transforms, lst_absurds)]
# with open(src_results, 'wb') as f:
#     pickle.dump(root, f)

In [ ]:
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swam_and_lake\2588_1168_lake\results\pyramid_transforms_test_with_absurd_vals.pickle"
# src_tile_original = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_7_different_max_correspondances\down_to_0.2\swisssurface3d_2018_2589-1169_2056_5728.ply"
# src_out_gpkg = os.path.join(os.path.dirname(src_transforms), 'points_translate.gpkg')

# tile_original = o3d.io.read_point_cloud(src_tile_original)
# z_mean = tile_original.compute_mean_and_covariance()[0][2]
# offset = [2589500, 1169500, z_mean]
# # tile_original.translate(np.array([-2589500, -1169500, 0]) - np.array([0, 0, z_mean]))

# with open(src_transforms, 'rb') as f:
#     lst_transforms = pickle.load(f)

# data = []
# bbox_data = []
# for _, el in tqdm(enumerate(lst_transforms), total=len(lst_transforms)):
#     el_lvl = el[0]
#     bbox_dict = el[1][0]
#     bbox_data.append(bbox_dict)
#     bbox = o3d.geometry.AxisAlignedBoundingBox(
#         min_bound=np.array(bbox_dict["min_bound"]),
#         max_bound=np.array(bbox_dict["max_bound"])
#     )

#     transform = el[1][3]

#     # translation:
#     # cropped_tile = tile_original.crop(bbox)
#     xmin, ymin, zmin = bbox_dict['min_bound']
#     xmax, ymax, zmax = bbox_dict['max_bound']
#     center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
#     # center = np.eye(4)
#     translated = np.linalg.matmul(transform, center)
#     norm = float(np.linalg.norm(translated - center))
#     if norm == 0:
#         print(xmin, ymin)
#         print(transform)
#         print('---')
#     norm2d = float(np.linalg.norm(translated[0:2] - center[0:2]))
#     direction = (translated[0:2] - center[0:2]) / norm2d 

#     data.append((center[0][0] + offset[0], center[1][0] + offset[1], center[2][0] + offset[2], norm, direction[0][0], direction[1][0], el[0], el[-2], el[-1]))

# columns = ['x', 'y', 'z', 'translation', 'translation_x', 'translation_y', 'lvl', 'is_leaf', 'absurd_status']

# # Export all tiles
# export_points_and_bboxes(
#     data=data,
#     bbox_data=bbox_data,
#     columns=columns,
#     output_path=src_out_gpkg,
#     offset=offset,
# )

# # Export only leaves
# mask_leaves = np.array([x[-2] for x in data], dtype=np.bool)
# data_leaves = list(np.array(data)[mask_leaves])
# bbox_data_leaves = list(np.array(bbox_data)[mask_leaves])

# export_points_and_bboxes(
#     data=data_leaves,
#     bbox_data=bbox_data_leaves,
#     columns=columns,
#     output_path=src_out_gpkg.split('.gpkg')[0] + "_leaves.gpkg",
#     offset=offset,
# )

TypeError: 'QuadNode' object is not iterable

In [ ]:
# # laoding of gpkg file
# # src_gpkg = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_5_min_points\3000_max_lvl_7\results\points_translate.gpkg"
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\pyramid_transforms_test.pickle"
# src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_10_swam_and_lake\2588_1168_lake\results\pyramid_transforms_test.pickle"
# src_results = src_transforms.split('.pickle')[0] + "_with_absurd_vals.pickle"
# with open(src_transforms, 'rb') as f:
#     lst_transforms = pickle.load(f)

# data['is_absurd'] = np.zeros(len(data), dtype=np.bool)
# lst_absurds = np.zeros(len(lst_transforms), dtype=np.bool)
# lst_absurds = [False] * len(lst_transforms)
# print(len(lst_absurds))

# for id_el, node in tqdm(enumerate(lst_transforms), total=len(lst_transforms)):
#     # print(id_el)
#     # node = el[0]
#     bbox = node.bbox
#     transform = node.transform
#     parent = node.parent
#     # lvl_el = node.level

#     if node.level == 0:
#         continue
    
#     # compare to parent value
#     val = get_translate_from_element(bbox, transform)
#     # id_parent = id_el-1
    
#     # while(lst_transforms[id_parent][0] != node.level-1):
#     #     id_parent -= 1

#     parent_val = get_translate_from_element(parent.bbox, parent.transform)
#     diff = abs(parent_val - val)
#     if diff > 10 or lst_absurds[id_parent] == True:
#         lst_absurds[id_el] = True

# print("Number of absurd values: ", np.sum(lst_absurds))
# lst_transforms = [(*x,y) for x,y in zip(lst_transforms, lst_absurds)]
# with open(src_results, 'wb') as f:
#     pickle.dump(root, f)